# Vantara — 03 Model Experiments

This notebook is a thin analysis consumer of STEP 04 evidence produced by `src.models.step04_pipeline`. It does not fit models or access the final held-out test.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.models.common import load_feature_schema  # noqa: E402

REPORTS = ROOT / 'reports/modeling'
schema = load_feature_schema(ROOT / 'models_artifacts/churn_feature_schema.json')
summary = json.loads((REPORTS / 'step04_summary.json').read_text())
{
    'schema_version': schema['schema_version'],
    'feature_count': schema['feature_count'],
    'held_out_test_accessed': summary['held_out_test_accessed'],
}

{'schema_version': 'vantara-churn-features-v1',
 'feature_count': 47,
 'held_out_test_accessed': False}

## Six classical churn models — training CV and validation evidence

In [2]:
churn = pd.read_csv(REPORTS / 'churn_model_comparison.csv')
display(churn[[
    'model', 'cv_roc_auc_mean', 'validation_accuracy', 'validation_precision',
    'validation_recall', 'validation_f1', 'validation_roc_auc',
    'validation_confusion_matrix'
]])

,model,cv_roc_auc_mean,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc,validation_confusion_matrix
0,logistic_regression,0.798004,0.727763,0.770202,0.733173,0.751232,0.803821,"[[235, 91], [111, 305]]"
1,random_forest,0.802289,0.730458,0.750000,0.778846,0.764151,0.801587,"[[218, 108], [92, 324]]"
2,xgboost,0.804554,0.726415,0.749415,0.769231,0.759193,0.793903,"[[219, 107], [96, 320]]"
3,decision_tree,0.775749,0.730458,0.763415,0.752404,0.757869,0.789051,"[[229, 97], [103, 313]]"
4,svm,0.782651,0.726415,0.726115,0.822115,0.771139,0.787426,"[[197, 129], [74, 342]]"
5,lightgbm,0.797641,0.726415,0.742597,0.783654,0.762573,0.785099,"[[213, 113], [90, 326]]"


## Predicted 180-Day Customer Value — Ridge and XGBRegressor

In [3]:
clv = pd.read_csv(REPORTS / 'clv_model_comparison.csv')
display(clv[[
    'model', 'cv_mae_mean', 'cv_rmse_mean', 'cv_r2_mean',
    'validation_mae', 'validation_rmse', 'validation_r2'
]])

,model,cv_mae_mean,cv_rmse_mean,cv_r2_mean,validation_mae,validation_rmse,validation_r2
0,ridge,746.839497,4090.805899,-2.179794,475.217661,1083.620615,0.968463
1,xgboost_regressor,633.509064,3073.337373,0.316998,626.006521,5693.909341,0.129261


## K-Means and GMM segmentation with business profiles and PCA

In [4]:
segmentation = pd.read_csv(REPORTS / 'segmentation_model_selection.csv')
profiles = pd.read_csv(REPORTS / 'segment_profiles.csv')
pca_sample = pd.read_csv(REPORTS / 'segmentation_pca_sample.csv')
display(segmentation)
display(profiles)
pca_sample.head()

,algorithm,components,silhouette,davies_bouldin,inertia,bic,training_seconds,mlflow_run_id,held_out_test_accessed
0,kmeans,3,0.259803,1.257543,25299.554648,NaN,1.129499,27d9e026a454453f8002e18d6f6445b3,False
1,kmeans,4,0.269508,1.188545,21853.263118,NaN,0.310599,ccadb140383b4b74974f1e21de6d2b9c,False
2,kmeans,5,0.271580,1.091255,19867.627572,NaN,0.265113,5de0684153b04854ba21cce493b2a6c4,False
3,kmeans,6,0.289979,1.071463,17907.701610,NaN,0.257989,b30c3bcb5f6e4a9599a61bf1cacd289b,False
4,kmeans,7,0.297376,1.069313,16103.418368,NaN,0.277221,02a9378bce2d40e28913c1c34218fd9e,False
5,kmeans,8,0.305245,1.100953,14576.835494,NaN,0.256407,5f777eb2e0fd43adb9021d15cb813c40,False
6,gmm,2,0.229324,2.057081,NaN,37928.495074,0.109099,ed3fcdcb15314909911c3fee6f7634b2,False
7,gmm,3,0.111261,2.601405,NaN,7565.501172,0.227901,87b346c8152f453c9b11a81e8b7a4a3c,False
8,gmm,4,0.101411,2.221703,NaN,-10325.738906,0.249050,14c6a9c529704a06b5352324503af31f,False
9,gmm,5,0.103250,2.297879,NaN,-19172.915873,0.250367,36b9dc7f889a42319841f71e7c73e106,False


,algorithm,kmeans_segment,recency_days,frequency_orders,net_spend,avg_order_value,avg_basket_units,purchase_frequency_trend,variance_interpurchase_gap_days,return_rate,markdown_affinity_proxy,seasonal_purchase_concentration,customer_count,business_label,gmm_segment
0,kmeans,0.0,52.182766,13.873315,5727.729272,412.058732,256.102512,-2.938302e-02,2021.166386,0.016250,0.401179,0.363163,371,Loyal High Value,NaN
1,kmeans,1.0,371.028545,1.702797,421.438942,261.986309,149.278177,-1.344809e-04,238.048172,0.010723,0.017221,0.892469,1144,Lowest Engagement,NaN
2,kmeans,2.0,138.068838,3.858209,1153.615746,306.846874,175.098721,-1.230113e-04,21684.622652,0.022019,0.295114,0.524209,134,Developing,NaN
3,kmeans,3.0,102.940284,8.016254,2817.810844,337.700337,197.798768,7.467764e-03,2401.981605,0.012282,0.261950,0.380273,1292,Established,NaN
4,kmeans,4.0,301.761493,2.720978,1602.520448,526.727734,321.296479,2.685705e-04,726.399944,0.018674,0.885318,0.762396,491,At Risk,NaN
5,kmeans,5.0,319.464106,2.250000,460.053125,646.985885,1203.562500,1.373626e-03,1279.694434,1.110568,0.375000,0.880208,16,Dormant,NaN
6,kmeans,6.0,348.799861,2.600000,14967.708000,14038.351733,35961.573333,0.000000e+00,872.460216,0.302209,0.866667,0.760000,5,Occasional,NaN
7,kmeans,7.0,10.312927,140.230769,140861.320000,1352.549871,762.949222,-7.185123e-03,63.236566,0.029289,0.695520,0.229845,13,Champions,NaN
8,gmm,NaN,34.953541,10.819549,3676.314617,350.003443,207.611161,7.105676e-04,1384.392887,0.004716,0.337468,0.344773,665,Loyal High Value,0.0
9,gmm,NaN,53.633294,5.539683,1474.995429,292.910817,169.151900,-2.476888e-03,8667.753571,0.039376,0.308006,0.466009,315,Developing,1.0


,partition,pca_1,pca_2,kmeans_segment,gmm_segment
0,train,0.657383,0.914094,4,3
1,train,1.360058,0.096748,3,3
2,train,-1.154911,0.109119,1,2
3,train,-0.045890,-0.289337,3,2
4,train,0.334160,0.194647,3,6


## Next-purchase-category LightGBM and most-popular baseline

In [5]:
next_category = pd.read_csv(REPORTS / 'next_category_evaluation.csv')
display(next_category[[
    'model', 'macro_f1', 'top_1_accuracy', 'top_3_accuracy',
    'training_rows', 'validation_rows', 'classes'
]])

,model,macro_f1,top_1_accuracy,top_3_accuracy,training_rows,validation_rows,classes
0,most_popular_category_baseline,0.019452,0.374233,0.552147,1504,326,31
1,lightgbm_multiclass,0.089842,0.331288,0.585890,1504,326,31


## Item-to-item recommender offline evaluation

In [6]:
recommender = pd.read_csv(REPORTS / 'recommender_evaluation.csv')
display(recommender[[
    'model', 'recall_at_5', 'hit_rate_at_5', 'catalog_coverage',
    'eligible_evaluation_customers', 'catalog_items'
]])

,model,recall_at_5,hit_rate_at_5,catalog_coverage,eligible_evaluation_customers,catalog_items
0,item_to_item_cosine,0.039674,0.353933,0.241258,2670,4569


## Local MLflow traceability and artifact reload evidence

In [7]:
mlflow_runs = pd.read_csv(REPORTS / 'mlflow_run_summary.csv')
reload_checks = json.loads((REPORTS / 'artifact_reload_smoke.json').read_text())
display(mlflow_runs[['run_id', 'run_name', 'model_family', 'status']])
display(pd.DataFrame(reload_checks))

,run_id,run_name,model_family,status
0,e87731945b07404da0b99b6df23d232c,churn_decision_tree,churn,FINISHED
1,8ff526e9ac5d4837a69a78ba321a7740,churn_lightgbm,churn,FINISHED
2,b2cdd7425dc2493fb8f4811f600d5719,churn_logistic_regression,churn,FINISHED
3,7a4e349cb7f7454794d9380d3b17c5e6,churn_random_forest,churn,FINISHED
4,6ba8166885854a42bb1eabd27dface31,churn_svm,churn,FINISHED
5,b82921356c1b488f89c9875a05fd9778,churn_xgboost,churn,FINISHED
6,c6dd597003ad441482934337e7598c2a,clv_ridge,clv,FINISHED
7,fed32e4475ef4ecca525c6c26c81ac63,clv_xgboost_regressor,clv,FINISHED
8,92d411e24a9f4200b499e5bf06ea9734,next_category_lightgbm,next_category,FINISHED
9,b91c4523e7744db3b86db04c82fe4f00,recommender_item_to_item,recommender,FINISHED


,artifact,reload,sample_output
0,churn_decision_tree.joblib,PASS,0.590860
1,churn_lightgbm.joblib,PASS,0.627240
2,churn_logistic_regression.joblib,PASS,0.448597
3,churn_random_forest.joblib,PASS,0.550014
4,churn_svm.joblib,PASS,0.699816
5,churn_xgboost.joblib,PASS,0.618088
6,clv_ridge.joblib,PASS,139.034870
7,clv_xgboost_regressor.joblib,PASS,98.523827
8,next_category_lightgbm.joblib,PASS,0.000000
9,segmentation_bundle.joblib,PASS,3.000000


## STEP 04 boundary

All tuning used training-only five-fold CV and all reported comparison metrics are validation evidence. Production churn selection, threshold freezing, explainability, and the one-time final held-out test remain deferred to STEP 06.